# 09a · 定版和考试

## 这一章在整个项目里的位置

```
00–08 ✅ → [09 上线] → 10 对外 API
              │
              ├─ 09a 定版和考试：调参、定版、打开考卷、和已发表模型比   ← 你在这里
              ├─ 09b 注册模型：模型登记进注册表，加一道门禁
              └─ 09c 全蛋白组打分：20,652 个蛋白的每一个可能突变
```

## 背景：为什么需要这一章

08 章定下了两个模型：稳定性用小网络，功能用三个模型的集成。到现在为止，所有成绩都来自**训练蛋白上的交叉验证**。那 108 个考试蛋白，从 07a 锁进抽屉以后一眼都没看过。

这一章做三件事：
1. **定版**：模型的最后一次调整。调参、用全部训练蛋白训练出最终的模型
2. **考试**：打开考卷，**只考这一次**
3. **同题对照**：在考试蛋白的同一批突变上，和旧 QAFI、AlphaMissense、EVE、popEVE、CPT-1 比

**为什么考试只能一次**：考完以后，只要你根据考试成绩再改一点模型、再考一次，考试蛋白就变成了"调参用的数据"，成绩就不再代表"从没见过的蛋白"。所以这一章的代码里有一道锁：考试记录一旦写进 MLflow，再运行那一格就会直接报错。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2 步：在 Databricks 上启动"旧预测全量装表"（约 1 小时），它在云上跑，你的 notebook 接着往下走
   ├─ 第 3、4 步：在你的 Mac 上调参、定版（约 30 分钟）
   ├─ 第 5 步：打开考卷（几分钟）
   └─ 第 6 步：等旧预测装好，同题对照
看结果：这里的输出；Databricks 网页的 Experiments 和 Jobs
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 上线的约束改变了模型 | 实验室的配方要能放大生产 |
| 2 | 旧预测全量装表 | 先把对照组的样品备齐 |
| 3 | 调参 | 正式定方案前，把条件摸一遍 |
| 4 | 定版 | 方案锁定，正式生产 |
| 5 | 考试 | 盲测，只测一次 |
| 6 | 同题对照 | 和市面上的产品同台比 |

**花费：0**。**时间**：约 1.5 小时，大部分是等。

---

## 准备

运行下面这一格。**不用看代码。** 第一行很要紧：这一章同时用 PyTorch（网络）和 XGBoost（树模型），07f 说过，两者在 Mac 上一起多线程运行会崩溃，所以先让它们都单线程。网络用 GPU，不受影响；树模型慢一点，训练一次一分钟以内。

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # PyTorch 和 XGBoost 在同一个进程里：只能单线程（见 07f）
import subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.models import infer_signature
from qafi_platform import lakehouse, tracking
from qafi_platform.lakehouse import sql
from qafi_core import baseline, neural, release, tuning
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
ESM2 = "08e4846e537177426273712802403f7ba8261b6c"
MODELS = ROOT / "data/models"
MODELS.mkdir(parents=True, exist_ok=True)
print("Databricks:", w.config.host)
print("考卷开过没有:", "开过了！" if len(tracking.test_set_opened(EXPERIMENT)) else "还没有")

### 结果怎么看
最后一行应该是"**还没有**"。如果是"开过了"，说明考试那一步已经做过，第 5 步会拒绝再做。

---

## 第 1 步 · 上线的约束改变了模型

### 在干什么
说明上线用的模型和 08a 那个为什么有两处不同。

### 背景
比喻：**实验室的配方要能放大生产**。在试管里能做的事，到了一万升的发酵罐里不一定做得了。

我们的"发酵罐"是 09c：给 20,652 个蛋白的**每一个可能突变**打分，一共约 **2.17 亿**个。这带来两个约束：

**约束一：每个残基只能算一次。** 08a 的网络吃"向量 + 16 个特征"。16 个特征里有 6 个描述的是**这次替换**（BLOSUM62、体积变化……），换一种氨基酸就要重算一遍，2.17 亿次。另外 10 个描述的是**这个残基**（pLDDT、rsa、熵……），和换成什么无关。网络本来就输出 20 个分数（每种氨基酸一个），所以只要输入全是残基级的，**每个残基算一次，20 种替换的分数一起出来**，1,140 万次就够。

我量过只用残基级输入会损失多少：

| 网络的输入 | 整体 | Domainome | ProteinGym | 功能 |
|---|---|---|---|---|
| 向量 + 全部 16 个特征（08a） | 0.627 | 0.651 | 0.634 | 0.378 |
| 向量 + 10 个残基级特征 | 0.625 | 0.651 | **0.658** | **0.392** |

**不亏，还略好一点。** 替换级的信息，网络从 20 个输出里自己就学到了。

**约束二：分数要能跨蛋白比。** 08a 的集成是在**每组实验内部**排名再平均。可是上线以后没有"实验"，用户可能拿 TP53 的一个突变和 BRCA1 的一个突变比。所以改成：每个成员先换成它在**全部训练突变**里的百分位，再取平均。

| 集成的做法 | 功能 |
|---|---|
| 每组实验内部排名（08a） | 0.432 |
| 全部训练突变里的百分位（上线用） | 0.428 |

配对比较 p = 0.91，没有差别。

**最后的分数长这样**：两个分数都在 0 到 1 之间，意思是"这个突变比训练集里百分之多少的突变更像野生型"。比如功能分 0.05 = 比 95% 的已知突变更可能破坏功能。

### 公司里什么情况怎么做
- **模型的设计要从上线倒推**。研究里随便用的特征，上线时可能算不起、拿不到或太慢。越早考虑，越少返工
- 这次运气好，约束没有让成绩变差。常常是会变差的，那就要权衡：多花多少钱，换多少成绩

### 面试怎么说
> "Two production constraints reshaped the model: inputs had to be residue-level so one forward pass gives all 20 substitutions, 11 million passes instead of 217 million, which cost nothing in cross-validation; and the ensemble averages global percentiles instead of within-assay ranks, so scores are comparable across proteins, again at no cost."

---

## 第 2 步 · 旧预测全量装表

### 在干什么
在 Databricks 上把旧 QAFI 全蛋白组预测（04 号放进 raw 的那个 7.4 GB 压缩包）全部装进 `silver.variant_scores`，第 6 步要用。

### 背景
04 号写好了这条流水线（`qafi old proteome predictions`），但当时只试了 20 个蛋白。这次装全部 2 万个，约 **1 小时**。

**写这一章时发现了一个数据问题**，已经修好，这里说一下：旧文件里同一个突变常常重复几十到几千行（拼接文件时的"扇出"）。照理说这些拷贝里的分数应该一模一样，在前 2,000 个蛋白里检查发现：
- QAFI、AlphaMissense、CPT-1：完全一致
- **EVE、popEVE**：0.4% 的突变，不同拷贝的值不一样。多半是不同转录本的分数被拼到了同一个位置上

说不清哪个对，就**当作不知道**：这些突变的 EVE、popEVE 记为空，并记下是哪几列冲突（`conflicting_columns`）。质量检查分成两条：QAFI、AlphaMissense、CPT-1 必须一致（不一致就让流水线失败）；EVE、popEVE 的冲突比例低于 1% 就只记一笔。

### 用了什么技术，为什么
这个任务在**云上**跑，你的 notebook 不用等它：用 Databricks 的 SDK 发一个"开始运行"的指令（`run_now`），马上返回，任务在 Databricks 那边自己跑。第 6 步再回来看它跑完没有。**云上的活和本机的活同时进行**，这是省时间的常见做法。

### 执行
先部署图纸（把修好的代码送上去），再启动：

In [ ]:
run("cd databricks && databricks bundle deploy")
job = next(w.jobs.list(name="qafi old proteome predictions"))
started = w.jobs.run_now(job_id=job.job_id, job_parameters={"limit": "0"})   # 0 = 全部蛋白
print("已启动，运行编号", started.response.run_id, "；在 Databricks 网页的 Jobs & Pipelines 里能看到它在跑")

### 结果怎么看
打印出一个运行编号就说明启动了。去 Databricks 网页 → **Jobs & Pipelines** → `qafi old proteome predictions`，能看到四个方框，第二个（`bronze_proteome`）最久。

---

## 第 3 步 · 调参

### 在干什么
用 Optuna 在**训练蛋白**上系统地找一组最好的网络设置。

### 背景
比喻：**正式定方案前，把条件摸一遍**。做实验前你会试几个温度、几个浓度，挑最好的那组。网络也有一堆要人定的数（第 08a 章说的"超参数"）：

| 超参数 | 搜索范围 | 意思 |
|---|---|---|
| `hidden` | 256、512、1024 | 第一层有多少"评委" |
| `dropout` | 0.1 – 0.4 | 训练时让多少评委请假 |
| `learning_rate` | 0.0001 – 0.001 | 每一步改多大 |
| `epochs` | 2 – 8 | 训练几轮 |
| `weight_decay` | 0.0001 – 0.1 | 让参数保持小一点的力度，防止死记硬背 |

一组设置叫一次**试验**（trial）。每次试验都完整地做一遍 5 折交叉验证，成绩是全部 445 组训练实验的 Spearman 中位数。

**Optuna 不是瞎试**：前几次随便试，之后它会根据已有的结果估计"哪一带的设置成绩好"，往那一带多试。这个方法叫 TPE（Tree-structured Parzen Estimator）。第一次试验固定是现在用的设置，所以找到的只会更好，不会更差。

**考试蛋白完全不参与**。这正是 07a 锁考卷的意义：调参可以随便调，只要只看训练蛋白。

### 用了什么技术，为什么
- **Optuna**：目前最常用的调参库，开源，和 PyTorch、XGBoost 都能配合
- **每次试验都记进 MLflow**，挂在一个"父运行"下面（嵌套运行）。在网页上打开 `optuna_stability_head`，能看到底下 15 个试验，可以按成绩排序、画图看哪个参数最要紧

### 执行
15 次试验，每次约 1.5 分钟，一共 **约 25 分钟**。先取训练数据和向量（和 08a 一样）：

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'train'", MODELS / "train_variants.parquet")
train = pd.read_parquet(MODELS / "train_variants.parquet")
VERSION = sql(w, f"DESCRIBE HISTORY {CATALOG}.gold.fct_variant_features LIMIT 1")["version"][0]
parts = ROOT / f"data/esm2-650m/{ESM2}-labelled/embeddings"
if not parts.exists():   # 换了电脑：从平台的表里取，训练和考试蛋白一起取
    parts = MODELS / "labelled_embeddings.parquet"
    lakehouse.download(w, f"SELECT uniprot_id, position, embedding FROM {CATALOG}.silver.residue_embedding", parts)
table, index = neural.embedding_table(parts, train.uniprot_id.unique(), MODELS / "train_embeddings.npy")
print(f"{len(train):,} 个训练突变，{train.uniprot_id.nunique()} 个蛋白；向量表 {table.shape}")

In [ ]:
with tracking.tuning_run(EXPERIMENT, "optuna_stability_head", chapter="09a") as (parent, log_trial):
    study = tuning.search(train, table, index, release.RESIDUE_FEATURES, n_trials=15, on_trial=log_trial)
BEST = study.best_params
trials = study.trials_dataframe()
trials[["number", "value", "params_hidden", "params_dropout", "params_learning_rate", "params_epochs", "params_weight_decay"]] \
    .rename(columns=lambda c: c.replace("params_", "")).sort_values("value", ascending=False).round(4).head(8)

### 结果怎么看
表里是成绩最好的 8 次试验，`value` 是交叉验证的 Spearman 中位数。

- 第 0 次（`number` = 0）是原来的设置，大约 0.625
- 最好的一次大约 **0.63**。我写这一章时试了 3 次，最好的是 0.631：比原来高 0.006
- 前几名的设置往往差别不小，成绩却很接近。08a 第 4 步说过：**这个网络对大多数设置不敏感**

所以调参带来的提升很小，这在意料之中。这一步的意义是：**能说出"参数是系统搜索过的"，而且有记录可查**。

### 公司里什么情况怎么做
- 调参的预算要和收益匹配。这里 15 次就够了；一个上亿收入的推荐模型，可能值得跑几千次
- 调参永远只看验证数据（我们是训练蛋白上的交叉验证），**绝不看测试数据**

### 面试怎么说
> "I tuned five hyperparameters with Optuna's TPE sampler, 15 trials, each a full grouped 5-fold cross-validation on training proteins only, with the current setting as the first trial. Every trial is a nested MLflow run. The gain was small, about 0.006, which matched the earlier sensitivity checks."

---

## 第 4 步 · 定版

### 在干什么
用最好的设置，在**全部** 349 个训练蛋白上训练出最终的模型，连同把输出变成百分位的"换算表"一起存进 MLflow。

### 背景
比喻：**方案锁定，正式生产**。交叉验证里的模型每个都少看了一折，只用来打分。真正上线的模型要看过**全部**训练数据。

要存的东西有三样：
1. **稳定性网络**：最好的设置，全部训练蛋白
2. **树模型**：07f 的设置，全部训练蛋白（它是功能集成的成员之一）
3. **换算表**：集成的三个成员（零样本、树、网络），各自在训练突变上的分布，存成 1,001 个分位点。上线时，任何一个新的输出都查这张表换成百分位

换算表用的是**交叉验证里的预测**，不是最终模型在训练数据上的预测：后者见过这些突变，会偏乐观，分布就不对了。

### 执行
先用最好的设置再做一次完整的交叉验证，记进 MLflow（约 2 分钟），同时得到换算表：

In [ ]:
REFERENCES = {name: tracking.read_scores(w, run) for name, run in
              [("zero_shot", "esm2_zero_shot"), ("baseline", "baseline"), ("mlp_standardised", "mlp_standardised")]}
cv = neural.cross_validate(train, table, index, features=release.RESIDUE_FEATURES, standardise=True, settings=BEST)
tracking.cv_run(w, EXPERIMENT, "stability_head", {"model": "network on ESM-2 650M + 10 residue features", **neural.SETTINGS, **BEST,
                "features": ",".join(release.RESIDUE_FEATURES), "feature_table_version": VERSION},
                baseline.dataset_scores(cv), references=REFERENCES, chapter="09a")
lakehouse.download(w, f"SELECT * FROM {CATALOG}.ml.cv_predictions WHERE run_name = 'baseline'", MODELS / "baseline_predictions.parquet")
trees_cv = pd.read_parquet(MODELS / "baseline_predictions.parquet")[[*baseline.KEY, "predicted"]].rename(columns={"predicted": "trees"})
members = cv.rename(columns={"predicted": "stability"}).merge(trees_cv, on=baseline.KEY)
TABLES = release.distributions(members)                      # 三个成员各自的 1,001 个分位点
in_cv = release.scores(members, TABLES)
tracking.cv_run(w, EXPERIMENT, "function_ensemble", {"model": "mean of global percentiles", "members": ",".join(release.FUNCTION_MEMBERS)},
                baseline.dataset_scores(in_cv.assign(predicted=in_cv.function_score)), references=REFERENCES, chapter="09a")
pd.DataFrame({"stability_head": baseline.summarise(baseline.dataset_scores(cv)),
              "function_ensemble": baseline.summarise(baseline.dataset_scores(in_cv.assign(predicted=in_cv.function_score)))}).T.round(3)

### 结果怎么看
两行交叉验证成绩（我写这一章时的数）：
- `stability_head`：整体约 **0.63**，Domainome 约 0.65
- `function_ensemble`：功能（`spearman_median_mavedb`）约 **0.43**，和 08a 一样

再训练最终的两个模型，存进 MLflow（约 1 分钟）：

In [ ]:
HEAD = neural.fit(train, table, index, features=release.RESIDUE_FEATURES, standardise=True, settings=BEST)
TREES = baseline.fit(train)
sample = release.features_frame(train.head(1000))
runs_of = {
    "stability": tracking.model_run(EXPERIMENT, "final_stability_head", {**neural.SETTINGS, **BEST, "features": ",".join(release.RESIDUE_FEATURES),
                                    "feature_table_version": VERSION, "esm2_revision": ESM2}, HEAD, "pytorch",
                                    tracking.tensor_signature({"embedding": 1280, "features": len(release.RESIDUE_FEATURES)}, 20),
                                    {"distributions": TABLES}, chapter="09a"),
    "trees": tracking.model_run(EXPERIMENT, "final_trees", {**baseline.PARAMS, "feature_table_version": VERSION}, TREES, "xgboost",
                                infer_signature(sample, TREES.predict(sample)), {"distributions": TABLES}, chapter="09a"),
}
runs_of

### 结果怎么看
两个模型地址（`models:/m-...`），分别来自 MLflow 运行 `final_stability_head` 和 `final_trees`。两个运行里都存了换算表（`distributions.json`），模型都带着"输入长什么样"的说明（签名），09b 注册时要用。

### 公司里什么情况怎么做
- **最终模型 ≠ 交叉验证里的模型**。前者用全部数据训练，后者只用来估计成绩。这一点面试常问
- 把输出变成百分位的换算表，是模型的一部分，要和模型存在一起。分开存，迟早有人拿新模型配旧表

### 面试怎么说
> "The released models are refit on all training proteins with the chosen settings; cross-validation only estimates their performance. The percentile tables come from out-of-fold predictions, not in-sample ones, and are stored with the models."

---

## 第 5 步 · 考试

### 在干什么
打开考卷：在 108 个从没碰过的考试蛋白上，给最终模型打分。**只做一次。**

### 背景
比喻：**盲测，只测一次**。你可以想象成药物的三期临床试验：方案、终点、统计方法全部提前定好，揭盲以后不能再改。

我们的"提前定好"：
- 模型：第 4 步存进 MLflow 的那两个，加上换算表
- 打分方法：每组实验一个 Spearman，和交叉验证完全一样
- **上线门槛**（下一章的门禁要用）。三条规则，都必须在考试蛋白上成立：

| 规则 | 意思 |
|---|---|
| 稳定性分在稳定性实验上**赢过零样本** | 训练出来的网络必须比不训练好 |
| 稳定性分在稳定性实验上**赢过树模型** | 复杂的网络必须比简单的基线好 |
| 功能分在功能实验上**赢过零样本** | 集成必须比不训练好 |

"赢过"的意思：超过一半的实验更好，**并且**配对 Wilcoxon 检验 p < 0.05。

**那道锁**：`tracking.test_run` 写入 MLflow 之前，先查有没有带 `test_set = opened` 标签的运行。有，就直接报错。考试记录里还写明了考的是哪两个模型，09b 只注册这两个。

### 执行
取考试蛋白的突变和向量，用最终模型打分（几分钟）：

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'test'", MODELS / "test_variants.parquet")
exam = pd.read_parquet(MODELS / "test_variants.parquet")
test_table, test_index = neural.embedding_table(parts, exam.uniprot_id.unique(), MODELS / "test_embeddings.npy")
exam["stability"] = neural.score(HEAD, exam, test_table, test_index, release.RESIDUE_FEATURES)
exam["trees"] = TREES.predict(release.features_frame(exam))
exam = release.scores(exam, TABLES)
PER_MODEL = release.test_scores(exam, {"stability_score": "stability_score", "function_score": "function_score",
                                       "esm_delta": "esm_delta", "trees": "trees"})
GATE = release.gate(PER_MODEL)
test_run_id = tracking.test_run(EXPERIMENT, PER_MODEL, GATE, models=runs_of, chapter="09a")   # 第二次运行这一格会在这里报错
print(f"{len(exam):,} 个考试突变，{exam.uniprot_id.nunique()} 个蛋白，{exam.dataset_id.nunique()} 组实验；记录在 MLflow 运行 {test_run_id}")
GATE

### 结果怎么看
最后的表是三条门禁规则：`datasets better`（考试实验里赢的比例）、`mean gain`（平均高多少）、`p_value`、`passed`。

**这一格的结果我也不知道**：写这一章时我没有碰考试蛋白，它的第一次运行就是你这一次。我只在一折训练蛋白上验证过代码能跑通。

**你的这一次考试**（2026-10-14，记录在 MLflow 的 `test_evaluation` 运行里；这一格的原始输出在提交时被清掉了）：108 个考试蛋白、22.8 万个突变，**三条门禁全过**：
| 规则 | 更好的实验占比 | 平均高多少 | p 值 |
|---|---|---|---|
| 稳定性分 > 零样本（稳定性实验） | 96% | +0.197 | < 0.0001 |
| 稳定性分 > 树模型（稳定性实验） | 84% | +0.073 | < 0.0001 |
| 功能分 > 零样本（功能实验） | 88% | +0.043 | 0.002 |

各个分数的 Spearman 中位数：稳定性分 0.612，功能分 0.593，树模型 0.550，ESM-2 零样本 0.409。

把考试成绩和交叉验证成绩放在一起看：

In [ ]:
cv_side = {"stability_score": baseline.dataset_scores(cv), "function_score": baseline.dataset_scores(in_cv.assign(predicted=in_cv.function_score)),
           "esm_delta": REFERENCES["zero_shot"], "trees": REFERENCES["baseline"]}
pd.concat({"交叉验证（训练蛋白）": pd.DataFrame({n: baseline.summarise(s) for n, s in cv_side.items()}).T,
           "考试（考试蛋白）": pd.DataFrame({n: baseline.summarise(s) for n, s in PER_MODEL.items()}).T}, axis=1).round(3)

### 结果怎么看
左边是交叉验证，右边是考试。怎么读：
- **考试成绩通常比交叉验证略低**，几个百分点以内属正常：调参时挑设置，多少会挑到"恰好适合训练蛋白"的那一组
- **如果低很多**（比如 0.1 以上），要怀疑交叉验证有泄漏，或者考试蛋白和训练蛋白差别太大（考试集里功能实验只有 7 个蛋白、16 组实验，数字会抖得厉害）
- 门禁没过，**不是改模型再考**，而是记下来、带着这个结论进 09b：不上线，或者有条件地上线

### 公司里什么情况怎么做
- 测试集的结果要**原样报告**，不管好坏。挑好看的报，是学术界和工业界都最常见的问题之一
- 测试集用过以后，下一轮开发要换新的测试数据（比如新发表的 DMS 实验），旧的这一份就成了"见过的"

### 面试怎么说
> "The test proteins were evaluated once, with the release rules fixed in advance: the stability score must beat zero-shot and the tree baseline on stability assays, the function score must beat zero-shot on functional assays, each with a paired Wilcoxon p below 0.05. The code refuses a second evaluation."

---

## 第 6 步 · 同题对照

### 在干什么
在考试蛋白的同一批突变上，把我们的两个分数和五个已发表的预测器放在一起比。

### 背景
比喻：**和市面上的产品同台比**。旧 QAFI 的全蛋白组预测文件里，除了 QAFI 自己，还带着 AlphaMissense、EVE、popEVE、CPT-1 的分数。第 2 步把它装进了平台。

**先统一方向**：有的预测器"越大越致病"（AlphaMissense、EVE、CPT-1），我们的标签是"越大越像野生型"。比之前要先翻过来。方向写在 `release.PUBLISHED` 里，我在训练蛋白上逐个确认过：翻过来以后，每个都和标签正相关。

**每个预测器只在它自己打过分的突变上比**：EVE、popEVE 覆盖的蛋白比较少，强行要求"所有预测器都有分"会把数据砍掉一大半。

**一定要说明的一点**：Domainome 的实验，我们训练时见过同类（别的蛋白的 Domainome），已发表的预测器没见过。在 Domainome 上我们赢，有**主场优势**。功能实验上的比较才更公平。

### 执行
**内核重启过的话**（比如隔天再来，或者 VS Code 关过）：第 5 步不能重跑（考卷只开一次，程序会拒绝），下面要用的 `exam` 也就没有了。运行这一格：它从注册表取回**考过试的那两个模型**，在考试蛋白上重新算出分数，并和 MLflow 里的考试记录逐项核对。它**不会**再登记一次"开卷"。刚运行完第 5 步就跳过这一格。

In [ ]:
# 第 5 步不能重跑（考卷只开一次）。内核重启过的话运行这一格：从注册表取回考过试的两个模型，
# 在考试蛋白上重新算分，并和 MLflow 里的考试记录核对。它不会再登记一次"开卷"。
from qafi_platform import registry
PASSED, RULES, TESTED, TEST_RUN = registry.gate_result(EXPERIMENT)
HEAD = registry.load(registry.full_name("stability"), flavor="pytorch")
TREES = registry.load(registry.full_name("trees"), flavor="xgboost")
TABLES = registry.distributions(registry.full_name("trees"))
parts = ROOT / f"data/esm2-650m/{ESM2}-labelled/embeddings"
if not parts.exists():
    parts = MODELS / "labelled_embeddings.parquet"
    lakehouse.download(w, f"SELECT uniprot_id, position, embedding FROM {CATALOG}.silver.residue_embedding", parts)
if not (MODELS / "test_variants.parquet").exists():
    lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'test'", MODELS / "test_variants.parquet")
exam = pd.read_parquet(MODELS / "test_variants.parquet")
test_table, test_index = neural.embedding_table(parts, exam.uniprot_id.unique(), MODELS / "test_embeddings.npy")
exam["stability"] = neural.score(HEAD, exam, test_table, test_index, release.RESIDUE_FEATURES)
exam["trees"] = TREES.predict(release.features_frame(exam))
exam = release.scores(exam, TABLES)
PER_MODEL = release.test_scores(exam, {"stability_score": "stability_score", "function_score": "function_score",
                                       "esm_delta": "esm_delta", "trees": "trees"})
recorded = tracking.test_set_opened(EXPERIMENT).iloc[0]
print(f"考试记录 {TEST_RUN}，门禁{'通过' if PASSED else '没过'}；{len(exam):,} 个考试突变，{exam.uniprot_id.nunique()} 个蛋白")
pd.DataFrame({name: {"重新算": baseline.summarise(scores)["spearman_median"],
                     "考试记录": recorded[f"metrics.{name}_spearman_median"]} for name, scores in PER_MODEL.items()}).T.round(4)

两列一模一样，说明取回的就是考过试的模型、算法也没变。

再看旧预测装好没有（第 2 步启动的，约 1 小时）：

In [ ]:
lakehouse.latest_run(w, "qafi old proteome predictions")

四个任务都是 `SUCCESS` 再往下。还在跑就等一会儿再运行这一格。

取考试蛋白的旧预测，比：

In [ ]:
proteins = ", ".join(repr(p) for p in exam.uniprot_id.unique())
lakehouse.download(w, f"""SELECT uniprot_id, position, ref_aa, alt_aa, has_conflict, {', '.join(release.PUBLISHED)}
                          FROM {CATALOG}.silver.variant_scores WHERE uniprot_id IN ({proteins})""", MODELS / "test_published.parquet")
published = pd.read_parquet(MODELS / "test_published.parquet")
versus = release.against_published(exam, {"stability": "stability_score", "function": "function_score"}, published)
versus

### 结果怎么看
每一行：一个已发表的预测器（`published`）、我们的一个分数（`ours`）、一类实验（`data_kind`），在共同的突变上：
- `ours_median` 和 `published_median`：两边的 Spearman 中位数
- `ours_better`：我们更好的实验占多少
- `p_value`：配对检验

**怎么读**：
- **Domainome 那几行**：预期我们大幅领先，但这有主场优势，不能拿来宣传
- **mavedb（功能）那几行**才是真正的较量。考试蛋白上的结果（我们的功能分）：

  | 对手 | 我们 | 对手 | 我们更好的实验 | 结论 |
  |---|---|---|---|---|
  | AlphaMissense | 0.488 | 0.481 | 44% | 打平（p = 0.86） |
  | CPT-1 | 0.488 | 0.505 | 56% | 打平（p = 0.53） |
  | 旧 QAFI2 | 0.488 | 0.490 | 60% | 打平（p = 0.06） |
  | EVE | 0.540 | 0.404 | 92% | **我们更好**（p = 0.003） |
  | popEVE | 0.540 | 0.360 | 92% | **我们更好**（p = 0.001） |

  （EVE、popEVE 只覆盖一部分蛋白，所以那两行是在更小的一批突变上比的，我们的中位数也不同。）
- **Domainome（稳定性）那几行**：我们 0.60–0.62，五个对手 0.34–0.46，全部显著领先（p < 0.001）。但这有主场优势，只能说"在我们擅长的那类数据上更好"
- **proteingym 那几行**只有 2–3 组实验，p 值都不显著，不看
- 一句话：**功能上和最好的模型打平，稳定性上明显领先**。再加上 09c 的 ClinVar（临床致病性上 AlphaMissense、CPT-1 领先约 0.04），这就是我们在竞品中的真实位置
- 功能训练数据只有 29 个蛋白，能和 AlphaMissense（用了几百万个人群变异的信息）打平已经不错；要超过它，下一步还是要更多功能数据

### 公司里什么情况怎么做
- 和竞品比，必须**同一批数据、同一种指标、方向统一**，并且把对自己有利的条件（主场优势）主动说出来
- 输给成熟的竞品不丢人。说清楚**在哪一块输、为什么、怎么追**，比一句"我们更好"有价值得多

### 面试怎么说
> "On the held-out proteins I compared our two scores with QAFI2, AlphaMissense, EVE, popEVE and CPT-1 on the variants each of them covers, with orientations unified and checked on training data. I report stability assays separately because our model was trained on that assay type and the others were not."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 09a tuning, released models and the one evaluation on the test proteins") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 约束 | 只用残基级输入；集成改用全局百分位 | 从上线倒推模型；这次没有代价 |
| 2 旧预测 | 全量装表；EVE、popEVE 的冲突记为空 | 说不清的就当不知道，并留下记录 |
| 3 调参 | Optuna，15 次，只看训练蛋白 | 提升小，但有据可查 |
| 4 定版 | 全部训练蛋白训练；换算表来自交叉验证 | 最终模型 ≠ 交叉验证的模型 |
| 5 考试 | 一次，规则提前定好，代码上锁 | 结果原样报告 |
| 6 对照 | 同一批突变，方向统一 | 主动说出主场优势 |

**一句话**：模型按上线的约束定了版，在没碰过的蛋白上考了唯一的一次，并和已发表的预测器同台比了。

## 面试题

1. **Why can the test set be used only once?**
   > "Any change made after looking at test results turns the test set into tuning data, and its score no longer estimates performance on unseen proteins. We fixed the models and the release rules first and made the code refuse a second evaluation."

2. **What is the difference between the cross-validated models and the released model?**
   > "Cross-validation trains one model per fold to estimate performance. The released model is refit on all training data with the chosen settings."

3. **How did production constraints change your model?**
   > "Proteome-wide scoring meant one pass per residue, so the network takes only residue-level inputs and outputs all 20 substitutions; and ensemble members are mapped to global percentiles so scores compare across proteins. Both changes cost nothing in cross-validation."

4. **How do you compare against published predictors fairly?**
   > "Same variants, same metric, same orientation, and disclose advantages such as having trained on the same assay type."

**下一步 09b · 注册模型**：把这两个模型登记进 Unity Catalog 的模型注册表，用这次考试的结果过一道门禁，过了才挂上"冠军"标签。